# Export tiles as Xarray

In [ ]:
import ee
import xarray as xr
import numpy as np
import dask.array as da
import zarr

import odc.geo.xr  # noqa: F401

In [ ]:
# Authenticate and initialize
# ee.Authenticate()
ee.Initialize(opt_url="https://earthengine-highvolume.googleapis.com")

In [ ]:
dataset = "ACA/reef_habitat/v2_0"
ic = ee.ImageCollection(ee.Image(dataset))

In [ ]:
# Region of interest. Eventually, we need to do all of -180 to 180 and -32 to 32
left = 140
bottom = -26
right = 180
top = 0

# Close to full resolution is 0.00005
res = 0.0005
transform = [res, 0, left, 0, -res, top]

ds = xr.open_dataset(
    ic,
    engine='ee',
    geometry=[left, bottom, right, top],
    projection=ee.Projection(
        crs="epsg:4326", transform=transform
    ),
    chunks={"time": 1, "lon": 2000, "lat": 2000},
).squeeze().drop_vars("time")

# # Clean up just the reef mask for now
# reef_mask = ds.reef_mask.astype("uint8")
# reef_mask = reef_mask.transpose("lat", "lon")  # Transposed because GEE
# reef_mask.odc.nodata = 0

# reef_mask

In [ ]:
ds.to_zarr("aca.zarr", consolidated=False, compute=True, write_empty_chunks=False)